# 01 - Data Inspection

## Midterm Big Data Pipeline

**Dataset:** `orders_huge_mixed_quality.csv`

### Objectives
- Inspect the input file size.
- Inspect the CSV schema.
- Display a small sample of the raw records.
- Preserve the original dataset without modification.

> The dataset is large, so all inspection operations must be streaming-based.
> The complete file must never be loaded into memory.

In [ ]:
from pathlib import Path
import csv
import json

In [2]:
PROJECT_ROOT = Path.cwd().parent

DATA_FILE = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "orders_huge_mixed_quality.csv"
)

SAMPLE_ROWS = 10

DATA_FILE

WindowsPath('g:/Semestr_7/Amaly/Big Data/lec 5/H.W.BigData0v.0.1/data/raw/orders_huge_mixed_quality.csv')

### التحقق من الملف

In [3]:
if not DATA_FILE.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_FILE}"
    )

print("Dataset found successfully.")
print(f"Path: {DATA_FILE}")

Dataset found successfully.
Path: g:\Semestr_7\Amaly\Big Data\lec 5\H.W.BigData0v.0.1\data\raw\orders_huge_mixed_quality.csv


## 1. File Size

The first inspection step is to determine the size of the input dataset.

The file size will later be used by the File Router to select between:

- Python Batch
- Apache Spark

The file itself is not loaded into memory.

In [4]:
file_size_bytes = DATA_FILE.stat().st_size
file_size_mb = file_size_bytes / (1024 ** 2)
file_size_gb = file_size_bytes / (1024 ** 3)

print(f"File name: {DATA_FILE.name}")
print(f"Size: {file_size_bytes:,} bytes")
print(f"Size: {file_size_mb:,.2f} MB")
print(f"Size: {file_size_gb:,.2f} GB")

File name: orders_huge_mixed_quality.csv
Size: 13,264,821,500 bytes
Size: 12,650.32 MB
Size: 12.35 GB


### اكتشاف Schema

## 2. CSV Schema

The schema is extracted from the CSV header only.

No data rows are loaded during this step.

In [10]:
with DATA_FILE.open(
    "r",
    encoding="utf-8-sig",
    newline=""
) as file:
    
    reader = csv.reader(file)
    header = next(reader)

print(f"Number of columns: {len(header)}")
print()

for index, column in enumerate(header, start=1):
    print(f"{index:2}. {column}")


Number of columns: 17

 1. order_id
 2. order_date
 3. status
 4. customer_id
 5. customer_name
 6. customer_phone
 7. customer_email
 8. city
 9. district
10. delivery_type
11. delivery_cost
12. payment_method
13. payment_status
14. payment_amount
15. currency
16. total_amount
17. items_json


### حفظ الـSchema في متغير



In [11]:
schema_columns = header

schema_columns

['order_id',
 'order_date',
 'status',
 'customer_id',
 'customer_name',
 'customer_phone',
 'customer_email',
 'city',
 'district',
 'delivery_type',
 'delivery_cost',
 'payment_method',
 'payment_status',
 'payment_amount',
 'currency',
 'total_amount',
 'items_json']

### عرض أول 10 سجلات

## 3. Raw Sample - First 10 Records

Only the first 10 records are inspected.

The records are read sequentially using Python's CSV reader.
The complete dataset is never loaded into memory.

In [12]:
with DATA_FILE.open(
    "r",
    encoding="utf-8-sig",
    newline=""
) as file:
    
    reader = csv.DictReader(file)

    for row_number, row in enumerate(reader, start=1):
        
        if row_number > SAMPLE_ROWS:
            break
            
        print(f"\n--- Record {row_number} ---")
        print(json.dumps(row, ensure_ascii=False, indent=2))


--- Record 1 ---
{
  "order_id": "طلب-100000",
  "order_date": "2025-02-24T21:29:00",
  "status": "مؤكد",
  "customer_id": "عميل-0",
  "customer_name": "محمد علي",
  "customer_phone": "702390941",
  "customer_email": "user141764@example.com",
  "city": "تعز",
  "district": "شعوب",
  "delivery_type": "سريع",
  "delivery_cost": "5000.0",
  "payment_method": "محفظة إلكترونية",
  "payment_status": "تم الدفع",
  "payment_amount": "769000.0",
  "currency": "YER",
  "total_amount": "769000.0",
  "items_json": "[{\"sku\":\"SKU-1010\",\"name\":\"هاتف سامسونج A54\",\"qty\":-2,\"unit_price\":183000.0,\"total\":549000.0},{\"sku\":\"SKU-1010\",\"name\":\"هاتف سامسونج A54\",\"qty\":1,\"unit_price\":215000.0,\"total\":215000.0}]"
}

--- Record 2 ---
{
  "order_id": "طلب-100001",
  "order_date": "2025-01-12T00:13:00",
  "status": "قيد الانتظار",
  "customer_id": "عميل-1",
  "customer_name": "علي حسين",
  "customer_phone": "714876334",
  "customer_email": "user392083@example.com",
  "city": "لحج",
  "

## ملاحظة تحليلية

## Initial Observations

The first sample shows that the dataset contains mixed-quality records.

Examples observed in the initial sample include:

- Normal numeric values.
- Arabic numerals in numeric fields.
- Missing `customer_id`.
- Invalid email values.
- Invalid `items_json`.
- Negative item quantities.
- Different date and text formats.
- Values that may require normalization before validation.

These observations are preliminary and do not represent the full dataset.

A streaming-based data profiling step will be performed next to measure the actual frequency of these cases across the dataset.